# Testing the output linearity of the different Qblox modules.

Data can be found in their respective folders.

This notebook as used to obtain the data. Different notebooks can be found in the respective folders which process and plot the data.

Author: Kyle MacRobbie

### Setup

In [1]:
# Imports
import time
import json
import matplotlib.pyplot as plt # type: ignore
import pyvisa # type: ignore
import numpy as np # type: ignore
import scipy # type: ignore
from numpy import random # type: ignore
from __future__ import annotations 
from typing import TYPE_CHECKING, Callable
from qcodes.instrument import find_or_create_instrument # type: ignore
from qblox_instruments import Cluster, ClusterType # type: ignore
if TYPE_CHECKING:
    from qblox_instruments.qcodes_drivers.module import Module # type: ignore

In [2]:
# Run to get cluster IP
!qblox-pnp list

Devices:
 - 192.168.137.2: cluster_mm 0.9.1 with name "cluster-mm" and serial number 00015_2251_003


In [3]:
# Connect to cluster
cluster_ip = "192.168.137.2"
cluster_name = "cluster0"
cluster = find_or_create_instrument(
    Cluster,
    recreate=True,
    name=cluster_name,
    identifier=cluster_ip,
    dummy_cfg=(
        {
            2: ClusterType.CLUSTER_QCM,
            4: ClusterType.CLUSTER_QRM,
            6: ClusterType.CLUSTER_QCM_RF,
        }
        if cluster_ip is None
        else None
    ),
)
cluster.led_brightness('medium') # Sets LED brightness on the modules. Options are 'low', 'medium' and 'high'

# Get modules, and connect to the QRM
def get_connected_modules(cluster: Cluster, filter_fn: Callable | None = None) -> dict[int, Module]:
    def checked_filter_fn(mod: ClusterType) -> bool:
        if filter_fn is not None:
            return filter_fn(mod)
        return True

    return {
        mod.slot_idx: mod for mod in cluster.modules if mod.present() and checked_filter_fn(mod)
    }
modules = get_connected_modules(cluster)
module = list(modules.values())[0]

cluster.led_brightness('medium')

# reset cluster and print cluster status
cluster.reset()
print(cluster.get_system_status())

Status: OKAY, Flags: NONE, Slot flags: NONE


In [4]:
# Define which modules will be used
modules
qcm_module = modules[2]
qrm_module = modules[4]
rf_module = modules[6]

In [5]:
# Check QCM
print("\nQCM: {}\nQRM: {}\nRF: {}".format(qcm_module.is_qcm_type, qcm_module.is_qrm_type, qcm_module.is_rf_type))


QCM: True
QRM: False
RF: False


In [6]:
# Check QRM
print("\nQCM: {}\nQRM: {}\nRF: {}".format(qrm_module.is_qcm_type, qrm_module.is_qrm_type, qrm_module.is_rf_type))


QCM: False
QRM: True
RF: False


In [7]:
# Check QCM-RF
print("\nQCM: {}\nQRM: {}\nRF: {}".format(rf_module.is_qcm_type, rf_module.is_qrm_type, rf_module.is_rf_type))


QCM: True
QRM: False
RF: True


### Connecting to oscilloscope

In [8]:
# Get a list of all connected devices
rm = pyvisa.ResourceManager()
rm.list_resources()

('ASRL3::INSTR', 'GPIB0::8::INSTR')

In [9]:
# Connect to the oscillosocpe
scope = rm.open_resource('GPIB0::8::INSTR')
print(scope.query('*IDN?'))

*IDN LECROY,WP715ZI,LCRY0716N47852,8.5.0



### End of setup

---
---

### Function to run the experiment for one iteration

In [11]:
def run_rabi_sequence(pulse_length, amp):

	waveforms = {
		"block": {		# Block to be modulated and played as the RF pulse in step 4.
			"data": [amp for i in range(pulse_length)],
			"index": 0
		}
	}

	acquisitions = {
    	"acq": {"num_bins": 1, "index": 0},
	}

	# RF sequence, also sets marker for viewing on the oscilloscope
	seq_rf = f"""
		  wait_sync	  4
		  set_mrk	  {0b1111}
		  upd_param	  {pulse_length}

		  set_mrk	  {0b0000}
		  upd_param	  4

		  stop
	"""

	seq_out = f"""
		wait_sync	4
		play		0,0,{pulse_length}
		stop
	"""

	seq_in = f"""
		wait_sync	4
		acquire		0,0,{pulse_length}
		stop
	"""

	# UPLOAD SEQUENCES
	sequence_rf = {
		"waveforms": {},
		"weights": {},
		"acquisitions": {},
		"program": seq_rf,
	}

	sequence_out = {
		"waveforms": waveforms,
		"weights": {},
		"acquisitions": {},
		"program": seq_out,
	}

	sequence_in = {
		"waveforms": {},
		"weights": {},
		"acquisitions": acquisitions,
		"program": seq_in,
	}

	rf_module.sequencer0.sequence(sequence_rf)
	qrm_module.sequencer0.sequence(sequence_out)
	#qrm_module.sequencer1.sequence(sequence_in)

	# Disconnect previous output connections outputs
	rf_module.disconnect_outputs()
	qrm_module.disconnect_outputs()

	qrm_module.sequencer0.connect_out0("I")
	# qrm_module.sequencer1.connect_acq_I("in0") # Acquire through first input

	# qrm_module.scope_acq_sequencer_select(1) # Configure scope mode
	#qrm_module.scope_acq_trigger_mode_path0("sequencer")

	# qrm_module.scope_acq_avg_mode_en_path0(True)

	rf_module.sequencer0.sync_en(True)	# Enable sync
	qrm_module.sequencer0.sync_en(True)
	# qrm_module.sequencer1.sync_en(True)

	rf_module.arm_sequencer(0)  # arm sequencer 0
	qrm_module.arm_sequencer(0)
	# qrm_module.arm_sequencer(1)

	cluster.start_sequencer()	# Run the sequence

	rf_module.stop_sequencer(0)
	# qrm_module.stop_sequencer(1)
	qrm_module.stop_sequencer(0)

	# qrm_module.get_acquisition_status(1) # Wait for the sequencer to stop with a timeout period of one minute.
	# qrm_module.store_scope_acquisition(1, 'acq') # Move acquisition data from temporary memory to acquisition list.
	# data = qrm_module.get_acquisitions(1) # Get acquisition list from instrument.

	print("QCM-RF sequencer 0: " + str(rf_module.get_sequencer_status(0)))
	print("QCM sequencer 0   : " + str(qrm_module.get_sequencer_status(0)))

	return None
	# return data

### Reset and configure oscilloscope

In [31]:
scope.write(f'TIME_DIV {2e-6} S')		# Set the time scale on all oscilloscope channels
scope.write(f'C2:VOLT_DIV {1} V')		# Set the voltage scale on channel 1
scope.write(f'C1:VOLT_DIV {0.2} V')		# Set the voltage scale on channel 1
scope.write(f'TRIG_DELAY {-0.8e-5}')	# Set the time scale left/right on the display
scope.write(f'TRIG_MODE SINGLE')		# Set the trigger mode to "single" in order to take one acquisition

18

### Defining frequency, durations and looping through the pulse sequences

In [32]:
# Experiment variables
rf_pulse_length_ = 15000	# ns	# Length of the RF pulse (step 4)

readout_data = run_rabi_sequence(rf_pulse_length_, 1.0)	# Run the sequence

QCM-RF sequencer 0: Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
QCM sequencer 0   : Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []


In [ ]:
readout = readout_data['acq']['acquisition']['scope']['path0']['data']
fig, ax0 = plt.subplots(nrows = 1, ncols = 1, figsize = (14, 4))
ax0.plot(readout)
ax0.set_xlabel("Time [ns]")
ax0.set_ylabel("Input [V]")
ax0.set_title(f"Input")
plt.show()

In [ ]:
name = f"out1_in0_amp1"
file = open(rf"C:\\Users\\BaughLaflamme\\Desktop\\Qblox Master Folder\\CSG QBlox code\\Kyle MacRobbie\\Output linearity tests\\QRM_QRM\\{name}.txt", "w")
data_text = ""
for point in readout:
	data_text += f"{str(point)}\n"
file.write(data_text)
file.close()

### Stopping

In [ ]:
cluster.stop_sequencer()

print("QCM-RF sequencer 0: " + str(rf_module.get_sequencer_status(0)))
print("QRM sequencer 1: " + str(qrm_module.get_sequencer_status(1)))
print("QRM sequencer 1: " + str(qrm_module.get_sequencer_status(0)))

cluster.reset()
print(cluster.get_system_status())